In [ ]:
from data.crossner_utils import read_bio_file
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from seqeval.metrics import f1_score

MODEL="numind/NuNER-v2.0"

DOMAIN="ai"   # ← cambia dominio aquí
BASE="crossner"

train_tokens, train_labels = read_bio_file(f"{BASE}/{DOMAIN}/train.txt")
dev_tokens, dev_labels = read_bio_file(f"{BASE}/{DOMAIN}/dev.txt")

# label mapping
unique=set(l for seq in train_labels for l in seq)
label_list=sorted(unique)

label2id={l:i for i,l in enumerate(label_list)}
id2label={i:l for l,i in label2id.items()}

tokenizer=AutoTokenizer.from_pretrained(MODEL)

def tokenize(tokens,labels):

    tok=tokenizer(tokens,
        truncation=True,
        is_split_into_words=True)

    word_ids=tok.word_ids()

    new_labels=[]
    prev=None

    for w in word_ids:

        if w is None:
            new_labels.append(-100)
        elif w!=prev:
            new_labels.append(label2id[labels[w]])
        else:
            new_labels.append(-100)

        prev=w

    tok["labels"]=new_labels
    return tok

train_ds=Dataset.from_dict({"tokens":train_tokens,"labels":train_labels})
dev_ds=Dataset.from_dict({"tokens":dev_tokens,"labels":dev_labels})

train_ds=train_ds.map(lambda x: tokenize(x["tokens"],x["labels"]))
dev_ds=dev_ds.map(lambda x: tokenize(x["tokens"],x["labels"]))

model=AutoModelForTokenClassification.from_pretrained(
    MODEL,
    num_labels=len(label_list),
    ignore_mismatched_sizes=True
)

args=TrainingArguments(
    "nunertest",
    per_device_train_batch_size=8,
    num_train_epochs=5,
    learning_rate=3e-5,
)

def compute_metrics(p):

    preds=np.argmax(p.predictions,axis=2)

    true=[]
    pred=[]

    for pr,la in zip(preds,p.label_ids):

        t=[]
        p_=[]

        for x,y in zip(pr,la):
            if y!=-100:
                t.append(id2label[y])
                p_.append(id2label[x])

        true.append(t)
        pred.append(p_)

    return {"f1":f1_score(true,pred)}

trainer=Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

FileNotFoundError: [Errno 2] No such file or directory: 'data/crossner/train.txt'